# Vista RAW dei segnali + Start Index Cell (SIC) + correzione PADME

Script NUOVO e autonomo (non tocca `WC_analisys_tool.ipynb` ne' `RawView.ipynb`). Legge i
file `.mid.gz` reali con `wclib.py` (stessa libreria usata da WC_analisys_tool), calibra una
tabella di piedistallo per-capacitor (metodo PADME) e la applica ai segnali mostrati, con il
valore di **SIC** (Start Index Cell) sovrapposto a ogni evento (bank `DGH0`, campo gia'
estratto da `wc.daq_dgz_full2header`, valido per schede identificate come `1742` nel bank
MIDAS). Display e selezione eventi sono limitati alla finestra 100-250 ns.

**Cos'e' il SIC** (rif. PADME, Georgiev et al., *"Offline noise calibration of the CAEN
V1742 ADCs at the PADME experiment"*, J. Phys. Conf. Ser. 2255 (2022) 012008): il DRS4 e'
un buffer circolare di 1024 capacitor per canale; il SIC indica in quale capacitor fisico
e' stato scritto il campione 0 di QUESTO evento. La relazione tra il campione `t` (indice
nella waveform registrata) e la posizione fisica del capacitor `p` e':

```
p = (t + SIC) mod 1024
```

Eventi diversi hanno SIC diverso -> lo stesso capacitor fisico (con la sua eventuale
risposta anomala/piedistallo diverso) cade a un campione diverso in ogni evento. Questo e'
esattamente cio' che la correzione sotto sfrutta: si costruisce una tabella di piedistallo
per capacitor fisico (non per campione), e la si sottrae a ogni evento usando il suo SIC per
sapere quale valore di piedistallo usa ciascun campione.

In [ ]:
%matplotlib inline
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
import wclib as wc

# ============================================================
# CONFIG
# ============================================================
RUN      = 1339
path     = '/Users/benussi/nmv-data/WC/WC26'                          # <- JupyterHub remoto
# path   = '/Users/benussi/Testbeam2026_WC_unified/Data_TB'           # <- locale
TAG      = ''
CLOUD    = False
VERBOSE  = False

# canali reali modulo singolo mod0 (vedi CLAUDE.md): ch_A=27, ch_B=26, ch_trig=17
CHANNELS = [27, 26, 17]
SIGNAL_CHANNELS = [27, 26]  # canali usati per decidere se l'evento "ha segnale" (solo PMT A/B, non trig)

N_EVENTS = 6                # quanti eventi CON SEGNALE mostrare (una figura ciascuno) — ignorato se EVENT_INDEX e' impostato
SHOW_CAPACITOR = True       # sovrappone p(t) = (t+SIC) mod 1024, in ns, su ogni segnale (asse Y secondario)
N_DROP_LAST_CELLS = 12      # scarta le ultime N celle temporali: spike noti, problema hardware V1742

SIG_THR_MV = -10.0          # soglia selezione: mostra solo eventi con segnale corretto <= soglia, su almeno uno dei SIGNAL_CHANNELS
MAX_SCAN   = 5000           # limite di sicurezza sul numero di eventi scansionati per trovare N_EVENTS con segnale
WINDOW_NS  = (50.0, 250.0)  # finestra segnale: stesso range di WC_analisys_tool.ipynb (DEBUG['wf_xlim_pmt_ns'])

# Se impostato, ignora la scansione/selezione e mostra SOLO l'evento con questo indice assoluto
# (1-based, un incremento per ogni evento con bank DGH0 dall'inizio del file — stessa convenzione
# di 'n_gen' in WC_analisys_tool.ipynb, cosi' i due script possono mostrare lo STESSO evento fisico
# impostando DEBUG['event_index'] = EVENT_INDEX in WC_analisys_tool.ipynb).
EVENT_INDEX = 1004          # es. 17 per l'evento assoluto #17; None = comportamento normale (scansione)

N_CALIB = 1000               # eventi usati per calibrare la tabella piedistallo E_p (metodo PADME)
N_VALID = 1000               # eventi successivi, disgiunti dalla calibrazione, usati per validare la correzione

ADC_BITS  = 12
V_PP_MV   = 1000.0
ADC_TO_MV = V_PP_MV / (2**ADC_BITS)
N_CELLS_DRS4 = 1024

mfile = wc.open_mid(run=RUN, path=path, cloud=CLOUD, tag=TAG, verbose=VERBOSE)
odb   = wc.get_bor_odb(mfile)
try:
    sf_mhz = float(odb.data['Configurations']['SamplingFrequency'])
    dt_ns  = 1000.0 / sf_mhz
except Exception:
    dt_ns = 0.4
print(f"Run {RUN} | dt_ns={dt_ns:.4f} ({1000.0/dt_ns:.0f} MS/s)")

## Calibrazione piedistallo per-capacitor (metodo PADME)

Costruita PRIMA della vista eventi qui sotto, cosi' quest'ultima puo' mostrare la correzione
applicata a un singolo evento reale (non solo l'effetto medio su tante tracce come nella
sezione di validazione più in basso).

Per ogni canale si costruisce una tabella di piedistallo per-capacitor `E_p[cella]` (metodo
della **mediana**, robusto rispetto a impulsi reali occasionali, analogo al metodo "Mode" del
paper PADME — Georgiev et al. 2022), sui primi `N_CALIB` eventi. La correzione poi applicata e':

```
S_cor(t) = S_meas(t) - E_p[ (t + SIC) mod 1024 ]
```

In [ ]:
import time

t0 = time.time()
mfile_calib = wc.open_mid(run=RUN, path=path, cloud=CLOUD, tag=TAG, verbose=VERBOSE)
cal_matrix = {ch: np.empty((N_CELLS_DRS4, N_CALIB), dtype=np.int16) for ch in CHANNELS}

n_ev = 0
for event in mfile_calib:
    if n_ev >= N_CALIB:
        break
    for bank_name, bank in event.banks.items():
        if 'DGH0' not in bank_name:
            continue
        header = wc.daq_dgz_full2header(bank, verbose=False)
        waveform = wc.daq_dgz_full2array(event.banks['DIG0'], header, verbose=False, pedestal_correction=False)
        n_ch = int(header.nchannels[0])
        n_sub = int(header.ntriggers[0])
        for isub in range(n_sub):
            if n_ev >= N_CALIB:
                break
            sic = int(header.SIC[0][isub])
            spc = len(waveform[isub * n_ch])
            cell_idx = (np.arange(spc) + sic) % N_CELLS_DRS4
            for ch in CHANNELS:
                wf = np.asarray(waveform[isub * n_ch + ch], dtype=np.int16)
                cal_matrix[ch][cell_idx, n_ev] = wf
            n_ev += 1
        break
print(f"Calibrazione E_p: {n_ev} eventi ({time.time()-t0:.1f}s)")

E_p = {ch: np.median(cal_matrix[ch], axis=1) for ch in CHANNELS}   # piedistallo per-capacitor, ADC counts
del cal_matrix

## Vista eventi con correzione PADME applicata

Ogni campione viene corretto individualmente: `S_cor(t) = S_meas(t) - E_p[(t+SIC) mod 1024]`,
usando la tabella `E_p` appena costruita. Gli eventi usati per la calibrazione vengono saltati
(non mostrati qui), per non usare gli stessi eventi sia per calibrare sia per "abbellire" la
correzione mostrata.

In [ ]:
mfile_disp = wc.open_mid(run=RUN, path=path, cloud=CLOUD, tag=TAG, verbose=VERBOSE)

CH_A, CH_B, CH_TRIG = 27, 26, 17   # stessa mappatura di WC_analisys_tool.ipynb (mod0)

n_shown = 0
n_scanned = 0
n_skipped = 0
n_abs = 0   # indice assoluto evento (1-based), stessa convenzione di n_gen in WC_analisys_tool.ipynb

for event in mfile_disp:
    if n_shown >= N_EVENTS or n_scanned >= MAX_SCAN:
        break
    for bank_name, bank in event.banks.items():
        if 'DGH0' not in bank_name:
            continue
        header = wc.daq_dgz_full2header(bank, verbose=False)
        waveform = wc.daq_dgz_full2array(event.banks['DIG0'], header, verbose=False, pedestal_correction=False)

        n_ch  = int(header.nchannels[0])
        n_sub = int(header.ntriggers[0])          # sotto-eventi nello stesso blocco DGH0/DIG0
        spc_full = len(waveform[0])
        spc   = spc_full - N_DROP_LAST_CELLS       # celle finali scartate (spike noti V1742)
        t_ns  = np.arange(spc) * dt_ns
        full_window_ns = N_CELLS_DRS4 * dt_ns
        mask_win = (t_ns >= WINDOW_NS[0]) & (t_ns <= WINDOW_NS[1])

        for isub in range(n_sub):
            if n_shown >= N_EVENTS or n_scanned >= MAX_SCAN:
                break

            n_abs += 1

            if EVENT_INDEX is None:
                if n_skipped < N_CALIB:      # salta gli eventi usati per calibrare E_p (nessuna sovrapposizione)
                    n_skipped += 1
                    continue
                n_scanned += 1
            elif n_abs != EVENT_INDEX:
                continue

            sic = int(header.SIC[0][isub]) if len(header.SIC) > 0 else None
            sic_t_ns = sic * dt_ns if sic is not None else None
            cell_idx = (np.arange(spc) + sic) % N_CELLS_DRS4 if sic is not None else None
            p_ns = cell_idx * dt_ns if cell_idx is not None else None
            wrap_idx = np.where(np.diff(p_ns) < 0)[0] if p_ns is not None else []

            # --- correzione PADME per-capacitor: S_cor(t) = S_meas(t) - E_p[cella] ---
            wf_corr_by_ch = {}
            has_signal = False
            for ch in CHANNELS:
                wf_adc = np.asarray(waveform[isub * n_ch + ch][:spc], dtype=float)
                wf_corr = (wf_adc - E_p[ch][cell_idx]) * ADC_TO_MV
                wf_corr_by_ch[ch] = wf_corr
                if ch in SIGNAL_CHANNELS:
                    if wf_corr[mask_win].min() <= SIG_THR_MV:
                        has_signal = True

            # in modalita' scansione (EVENT_INDEX=None) si mostra solo se ha segnale;
            # in modalita' indice assoluto si mostra SEMPRE l'evento richiesto
            if EVENT_INDEX is None and not has_signal:
                continue

            # --- layout a 3 pannelli, stile WC_analisys_tool.ipynb ---
            fig, axes = plt.subplots(1, 3, figsize=(21, 5))

            # Pannello 1: PMT Signals (A+B sovrapposti, rosso/nero, come in WC_analisys_tool)
            ax0 = axes[0]
            ax0.plot(t_ns[mask_win], wf_corr_by_ch[CH_A][mask_win], color='red',   lw=1.5, label=f'PMT A (ch {CH_A})')
            ax0.plot(t_ns[mask_win], wf_corr_by_ch[CH_B][mask_win], color='black', lw=1.5, label=f'PMT B (ch {CH_B})')
            ax0.axhline(SIG_THR_MV, color='gray', ls=':', alpha=0.6)
            ax0.axhline(0, color='gray', lw=0.8, ls='--')
            if sic_t_ns is not None and WINDOW_NS[0] <= sic_t_ns <= WINDOW_NS[1]:
                ax0.axvline(sic_t_ns, color='navy', ls='--', lw=1.2, alpha=0.8, label=f'SIC = {sic_t_ns:.1f} ns')
            for wi in wrap_idx:
                if WINDOW_NS[0] <= t_ns[wi] <= WINDOW_NS[1]:
                    ax0.axvline(t_ns[wi], color='red', ls='--', lw=1.0, alpha=0.5)
            if SHOW_CAPACITOR and p_ns is not None:
                ax0_cap = ax0.twinx()
                ax0_cap.plot(t_ns[mask_win], p_ns[mask_win], lw=0.9, color='teal', alpha=0.6)
                ax0_cap.set_ylim(0, full_window_ns)
                ax0_cap.tick_params(axis='y', labelsize=7, colors='teal')
                ax0_cap.set_ylabel("cap [ns eq.]", fontsize=8, color='teal')
            ax0.set_title("PMT Signals (correzione PADME)")
            ax0.set_xlabel("Time (ns)"); ax0.set_ylabel("Amplitude (mV)")
            ax0.set_xlim(*WINDOW_NS)
            ax0.grid(True, ls='--', alpha=0.5); ax0.legend(loc='best', fontsize=8)

            # Pannello 2: Trigger
            ax1 = axes[1]
            ax1.plot(t_ns[mask_win], wf_corr_by_ch[CH_TRIG][mask_win], color='darkorange', lw=1.5,
                     label=f'Trigger (ch {CH_TRIG})')
            ax1.axhline(0, color='gray', lw=0.8, ls='--')
            if sic_t_ns is not None and WINDOW_NS[0] <= sic_t_ns <= WINDOW_NS[1]:
                ax1.axvline(sic_t_ns, color='navy', ls='--', lw=1.2, alpha=0.8)
            for wi in wrap_idx:
                if WINDOW_NS[0] <= t_ns[wi] <= WINDOW_NS[1]:
                    ax1.axvline(t_ns[wi], color='red', ls='--', lw=1.0, alpha=0.5)
            if SHOW_CAPACITOR and p_ns is not None:
                ax1_cap = ax1.twinx()
                ax1_cap.plot(t_ns[mask_win], p_ns[mask_win], lw=0.9, color='teal', alpha=0.6)
                ax1_cap.set_ylim(0, full_window_ns)
                ax1_cap.tick_params(axis='y', labelsize=7, colors='teal')
                ax1_cap.set_ylabel("cap [ns eq.]", fontsize=8, color='teal')
            ax1.set_title("Trigger (correzione PADME)")
            ax1.set_xlabel("Time (ns)"); ax1.set_ylabel("Amplitude (mV)")
            ax1.set_xlim(*WINDOW_NS)
            ax1.grid(True, ls='--', alpha=0.5); ax1.legend(loc='best', fontsize=8)

            # Pannello 3: placeholder (al posto del calorimetro, non serve per questo confronto)
            ax2 = axes[2]
            ax2.set_xlim(*WINDOW_NS)
            ax2.set_title("Calorimeter (placeholder)")
            ax2.set_xlabel("Time (ns)")
            ax2.text(0.5, 0.5, "placeholder", transform=ax2.transAxes,
                     ha='center', va='center', fontsize=12, color='gray')
            ax2.grid(True, ls='--', alpha=0.5)

            fig.suptitle(f"Run {RUN} - event {n_shown} | AbsIdx: {n_abs} (correzione PADME per-capacitor, finestra "
                         f"{WINDOW_NS[0]:.0f}-{WINDOW_NS[1]:.0f} ns) — SIC = {sic_t_ns:.1f} ns",
                         fontsize=12)
            plt.tight_layout()
            plt.show()
            plt.close(fig)

            n_shown += 1
            if EVENT_INDEX is not None:
                break
        if EVENT_INDEX is not None and n_shown > 0:
            break
        break   # una sola bank DGH0 per evento MIDAS
    if EVENT_INDEX is not None and n_shown > 0:
        break

if EVENT_INDEX is not None:
    if n_shown == 0:
        print(f"\nATTENZIONE: evento assoluto #{EVENT_INDEX} non trovato (il file ha solo {n_abs} eventi con bank DGH0).")
    else:
        print(f"\nMostrato evento assoluto #{EVENT_INDEX}.")
else:
    if n_shown < N_EVENTS:
        print(f"\nATTENZIONE: trovati solo {n_shown}/{N_EVENTS} eventi con segnale dopo aver scansionato {n_scanned} eventi (MAX_SCAN={MAX_SCAN}).")
    print(f"\nSaltati {n_skipped} eventi di calibrazione, scansionati {n_scanned}, mostrati {n_shown} eventi con segnale "
          f"(soglia {SIG_THR_MV} mV dopo correzione PADME, finestra {WINDOW_NS} ns, canali {SIGNAL_CHANNELS}).")

## Validazione della correzione (statistica su molti eventi)

Riusa la tabella `E_p` costruita sopra (primi `N_CALIB` eventi) e la applica a `N_VALID`
eventi successivi, **mai usati nella calibrazione**, per verificare se lo spread per-capacitor
si riduce davvero su dati indipendenti (altrimenti il confronto sarebbe drogato dall'aver
ottimizzato la correzione sugli stessi dati su cui la si misura).

In [ ]:
mfile_valid = wc.open_mid(run=RUN, path=path, cloud=CLOUD, tag=TAG, verbose=VERBOSE)

# ---- VALIDAZIONE su N_VALID eventi successivi ai N_CALIB usati per costruire E_p sopra ----
sum_raw  = {ch: np.zeros(N_CELLS_DRS4) for ch in CHANNELS}
sum_corr = {ch: np.zeros(N_CELLS_DRS4) for ch in CHANNELS}
cnt      = {ch: np.zeros(N_CELLS_DRS4) for ch in CHANNELS}

n_ev = 0
n_skipped = 0
for event in mfile_valid:
    if n_ev >= N_VALID:
        break
    for bank_name, bank in event.banks.items():
        if 'DGH0' not in bank_name:
            continue
        header = wc.daq_dgz_full2header(bank, verbose=False)
        waveform = wc.daq_dgz_full2array(event.banks['DIG0'], header, verbose=False, pedestal_correction=False)
        n_ch = int(header.nchannels[0])
        n_sub = int(header.ntriggers[0])
        for isub in range(n_sub):
            if n_ev >= N_VALID:
                break
            if n_skipped < N_CALIB:      # salta gli eventi gia' usati per costruire E_p
                n_skipped += 1
                continue
            sic = int(header.SIC[0][isub])
            spc = len(waveform[isub * n_ch])
            cell_idx = (np.arange(spc) + sic) % N_CELLS_DRS4
            for ch in CHANNELS:
                wf = np.asarray(waveform[isub * n_ch + ch], dtype=float)
                vals_by_cell = np.empty(N_CELLS_DRS4)
                vals_by_cell[cell_idx] = wf
                sum_raw[ch]  += vals_by_cell
                sum_corr[ch] += vals_by_cell - E_p[ch]
                cnt[ch]      += 1
            n_ev += 1
        break
print(f"Validazione: {n_ev} eventi indipendenti dalla calibrazione (dopo {n_skipped} eventi di calibrazione saltati)\n")

mean_raw_mv, mean_corr_mv = {}, {}
print(f"{'ch':<4}{'spread RAW (mV)':<18}{'std RAW (mV)':<16}{'spread CORR (mV)':<20}{'std CORR (mV)':<16}{'riduzione std':<12}")
for ch in CHANNELS:
    mean_raw_mv[ch]  = (sum_raw[ch] / cnt[ch]) * ADC_TO_MV
    mean_corr_mv[ch] = (sum_corr[ch] / cnt[ch]) * ADC_TO_MV
    spread_raw, std_raw   = mean_raw_mv[ch].max() - mean_raw_mv[ch].min(), mean_raw_mv[ch].std()
    spread_corr, std_corr = mean_corr_mv[ch].max() - mean_corr_mv[ch].min(), mean_corr_mv[ch].std()
    riduz = std_raw / std_corr if std_corr > 0 else float('nan')
    print(f"{ch:<4}{spread_raw:<18.2f}{std_raw:<16.3f}{spread_corr:<20.2f}{std_corr:<16.3f}{riduz:<12.1f}x")

In [ ]:
PLOT_CHANNELS = [27, 26, 17]   # ch A (grande recupero), ch B (grande recupero), ch trig (vero segnale, non migliora)

fig, axes = plt.subplots(len(PLOT_CHANNELS), 1, figsize=(9, 2.8 * len(PLOT_CHANNELS)), sharex=True)
for i, ch in enumerate(PLOT_CHANNELS):
    ax = axes[i]
    ax.plot(mean_raw_mv[ch], lw=0.8, color='crimson', label='RAW (media per capacitor)')
    # riallineo la curva corretta sulla PROPRIA media prima di sovrapporla: E_p e' basato sulla
    # mediana per-capacitor, mentre mean_raw_mv e' una media -> su canali con distribuzione
    # asimmetrica (es. ch17/trigger) mediana e media differiscono molto, e sommare semplicemente
    # mean_raw_mv[ch].mean() introdurrebbe un offset verticale spurio nel confronto visivo.
    corr_aligned = mean_corr_mv[ch] - mean_corr_mv[ch].mean() + mean_raw_mv[ch].mean()
    ax.plot(corr_aligned, lw=0.8, color='seagreen', label='corretto (riallineato sulla propria media)')
    ax.set_title(f"ch {ch}", fontsize=10)
    ax.set_ylabel("mV")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='upper right')
axes[-1].set_xlabel("cella fisica p (0-1023)")
fig.suptitle(f"Profilo per-capacitor prima/dopo correzione (validazione, {N_VALID} eventi indipendenti)",
             fontsize=11)
plt.tight_layout()
plt.show()

## Studio alternativo: spettro FFT medio, calo presente vs assente

Per i canali A e B (segnale corretto PADME), su tutta la finestra temporale disponibile
(nessuna soglia sui segnali): si calcola la FFT di ogni evento, e si media l'ampiezza
|FFT(f)| su tutti gli eventi, separatamente per eventi con calo presente e senza calo
(stesso criterio di validita' calo di `WC_analisys_tool.ipynb`: baseline interpolata tra
`CALO_BL_LEFT_NS`/`CALO_BL_RIGHT_NS`, poi `min(calo) <= CALO_THR`).

Non e' un conteggio di superamenti soglia — e' lo spettro medio di potenza sulla popolazione
di eventi, cosi' si vede direttamente quali frequenze sono legate al segnale fisico (piu'
presenti quando c'e' calo) rispetto a quelle sempre presenti (rumore elettronico/pickup,
indipendente dal calo).

In [ ]:
CH_CALO = 16
CALO_THR = -40.0
CALO_BL_LEFT_NS  = (0.0, 100.0)
CALO_BL_RIGHT_NS = (200.0, 400.0)
PMT_BL_LEFT_NS   = (0.0, 100.0)
PMT_BL_RIGHT_NS  = (300.0, 400.0)
N_FFT_EVENTS = 2000   # eventi da scansionare per lo studio FFT
N_DROP_LAST_FFT = 20  # celle finali escluse dalla finestra FFT (specifico per questo studio, non N_DROP_LAST_CELLS)

SHOW_RAW_FFT = False  # se True, calcola e mostra anche lo spettro FFT senza correzione PADME (per ora disattivato)

mfile_fft = wc.open_mid(run=RUN, path=path, cloud=CLOUD, tag=TAG, verbose=VERBOSE)

WINDOWS = ['rect', 'hann']   # 'rect' = nessuna finestra (rettangolare) | 'hann' = finestra di Hann
CORR_MODES = ['padme', 'raw']   # 'padme' = corretto per-capacitor | 'raw' = segnale grezzo, non corretto
sum_mag = {cm: {ch: {w: {'calo': None, 'nocalo': None} for w in WINDOWS} for ch in (CH_A, CH_B)} for cm in CORR_MODES}
n_calo, n_nocalo = 0, 0
freqs_mhz = None
_hann_cache = {}

def _interp_baseline(wf, t_ns, left_ns, right_ns, dt_ns):
    l0, l1 = int(round(left_ns[0]/dt_ns)),  int(round(left_ns[1]/dt_ns))
    r0, r1 = int(round(right_ns[0]/dt_ns)), int(round(right_ns[1]/dt_ns))
    med_l, med_r = np.median(wf[l0:l1]), np.median(wf[r0:r1])
    tc_l, tc_r = 0.5*(left_ns[0]+left_ns[1]), 0.5*(right_ns[0]+right_ns[1])
    slope = (med_r - med_l) / (tc_r - tc_l)
    return med_l + slope*(t_ns - tc_l)

n_ev = 0
for event in mfile_fft:
    if n_ev >= N_FFT_EVENTS:
        break
    for bank_name, bank in event.banks.items():
        if 'DGH0' not in bank_name:
            continue
        header = wc.daq_dgz_full2header(bank, verbose=False)
        waveform_padme = wc.daq_dgz_full2array(event.banks['DIG0'], header, verbose=False)
        waveform_raw   = (wc.daq_dgz_full2array(event.banks['DIG0'], header, verbose=False, pedestal_correction=False)
                          if SHOW_RAW_FFT else None)
        n_ch = int(header.nchannels[0]); n_sub = int(header.ntriggers[0])
        for isub in range(n_sub):
            if n_ev >= N_FFT_EVENTS:
                break
            n_ev += 1
            spc_full = len(waveform_padme[isub*n_ch])
            spc = spc_full - N_DROP_LAST_FFT
            t_ns = np.arange(spc) * dt_ns

            if spc not in _hann_cache:
                _hann_cache[spc] = np.hanning(spc)
            hann_win = _hann_cache[spc]

            # --- classificazione calo (sempre su segnale PADME-corretto, stesso metodo di WC_analisys_tool) ---
            wf_calo = np.asarray(waveform_padme[isub*n_ch+CH_CALO][:spc], dtype=float)
            bl_calo = _interp_baseline(wf_calo, t_ns, CALO_BL_LEFT_NS, CALO_BL_RIGHT_NS, dt_ns)
            is_calo_valid = np.min(wf_calo - bl_calo) <= CALO_THR
            grp = 'calo' if is_calo_valid else 'nocalo'
            if is_calo_valid: n_calo += 1
            else: n_nocalo += 1

            _modes = [('padme', waveform_padme)] + ([('raw', waveform_raw)] if SHOW_RAW_FFT else [])
            for cm, waveform in _modes:
                for ch in (CH_A, CH_B):
                    wf = np.asarray(waveform[isub*n_ch+ch][:spc], dtype=float)
                    bl = _interp_baseline(wf, t_ns, PMT_BL_LEFT_NS, PMT_BL_RIGHT_NS, dt_ns)
                    sig = wf - bl
                    mag_rect = np.abs(np.fft.rfft(sig))
                    mag_hann = np.abs(np.fft.rfft(sig * hann_win))
                    if freqs_mhz is None:
                        freqs_mhz = np.fft.rfftfreq(spc, d=dt_ns) * 1000.0   # cicli/ns -> MHz
                    for wname, mag in (('rect', mag_rect), ('hann', mag_hann)):
                        if sum_mag[cm][ch][wname][grp] is None:
                            sum_mag[cm][ch][wname][grp] = np.zeros_like(mag)
                        sum_mag[cm][ch][wname][grp] += mag
        break

print(f"Eventi analizzati: {n_ev}  (calo presente: {n_calo}, calo assente: {n_nocalo})")

_modes_to_plot = [('padme', 'PADME-corretto')] + ([('raw', 'grezzo (non corretto)')] if SHOW_RAW_FFT else [])
for cm, cm_label in _modes_to_plot:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
    for row, wname, wlabel in [(0, 'rect', 'nessuna finestra (rettangolare)'), (1, 'hann', 'finestra di Hann')]:
        for col, ch, lbl in [(0, CH_A, 'A'), (1, CH_B, 'B')]:
            ax = axes[row][col]
            mean_calo   = (sum_mag[cm][ch][wname]['calo']   / n_calo)   if (sum_mag[cm][ch][wname]['calo']   is not None and n_calo   > 0) else None
            mean_nocalo = (sum_mag[cm][ch][wname]['nocalo'] / n_nocalo) if (sum_mag[cm][ch][wname]['nocalo'] is not None and n_nocalo > 0) else None
            if mean_calo is not None:
                ax.plot(freqs_mhz, mean_calo, color='crimson', lw=1.2, label=f'calo presente (n={n_calo})')
            if mean_nocalo is not None:
                ax.plot(freqs_mhz, mean_nocalo, color='steelblue', lw=1.2, label=f'calo assente (n={n_nocalo})')
            ax.set_yscale('log')
            ax.set_title(f'ch {lbl} — {wlabel}', fontsize=10)
            if row == 1:
                ax.set_xlabel('Frequenza (MHz)')
            if col == 0:
                ax.set_ylabel('|FFT| medio (mV eq.)')
            ax.grid(True, which='both', alpha=0.3); ax.legend(fontsize=8)
    fig.suptitle(f'Run {RUN} — Spettro FFT medio ({cm_label}): calo presente vs assente', fontsize=12)
    plt.tight_layout()
    plt.show()

## Test: la potenza FFT in bassa frequenza come discriminante calo presente/assente

Verifica se un'unica variabile scalare per evento — la potenza integrata di `|FFT|^2` nella
banda 0-300 MHz (dove lo spettro medio separa nettamente calo presente da assente) — separa
bene le due popolazioni, come possibile alternativa/complemento alla soglia in ampiezza nel
tempo (`SIG_THR_A/B`).

Metriche riportate: istogrammi (scala log) delle due distribuzioni, **AUC** (0.5 = nessuna
separazione, 1.0 = separazione perfetta, via Mann-Whitney U) e accuratezza del miglior taglio
singolo su questa variabile.

In [ ]:
from scipy.stats import mannwhitneyu

LF_BAND_MHZ = (0.0, 300.0)   # banda dove lo spettro medio calo/no-calo si separa (vedi cella sopra)
N_LFPOWER_EVENTS = 2716      # eventi da scansionare (tutti quelli disponibili nel run)

mfile_lfp = wc.open_mid(run=RUN, path=path, cloud=CLOUD, tag=TAG, verbose=VERBOSE)

lfpower = {CH_A: {'calo': [], 'nocalo': []}, CH_B: {'calo': [], 'nocalo': []}}

n_ev = 0
for event in mfile_lfp:
    if n_ev >= N_LFPOWER_EVENTS:
        break
    for bank_name, bank in event.banks.items():
        if 'DGH0' not in bank_name:
            continue
        header = wc.daq_dgz_full2header(bank, verbose=False)
        waveform = wc.daq_dgz_full2array(event.banks['DIG0'], header, verbose=False)  # PADME-corretto
        n_ch = int(header.nchannels[0]); n_sub = int(header.ntriggers[0])
        for isub in range(n_sub):
            if n_ev >= N_LFPOWER_EVENTS:
                break
            n_ev += 1
            spc = len(waveform[isub*n_ch]) - N_DROP_LAST_FFT
            t_ns = np.arange(spc) * dt_ns

            wf_calo = np.asarray(waveform[isub*n_ch+CH_CALO][:spc], dtype=float)
            bl_calo = _interp_baseline(wf_calo, t_ns, CALO_BL_LEFT_NS, CALO_BL_RIGHT_NS, dt_ns)
            is_calo_valid = np.min(wf_calo - bl_calo) <= CALO_THR
            grp = 'calo' if is_calo_valid else 'nocalo'

            freqs = np.fft.rfftfreq(spc, d=dt_ns) * 1000.0
            band_mask = (freqs >= LF_BAND_MHZ[0]) & (freqs <= LF_BAND_MHZ[1])

            for ch in (CH_A, CH_B):
                wf = np.asarray(waveform[isub*n_ch+ch][:spc], dtype=float)
                bl = _interp_baseline(wf, t_ns, PMT_BL_LEFT_NS, PMT_BL_RIGHT_NS, dt_ns)
                sig = wf - bl
                mag = np.abs(np.fft.rfft(sig))
                power = float(np.sum(mag[band_mask]**2))
                lfpower[ch][grp].append(power)
        break

print(f"Eventi analizzati: {n_ev}")
for ch in (CH_A, CH_B):
    print(f"  ch {ch}: calo presente n={len(lfpower[ch]['calo'])}, calo assente n={len(lfpower[ch]['nocalo'])}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, ch, lbl in [(axes[0], CH_A, 'A'), (axes[1], CH_B, 'B')]:
    calo_vals = np.array(lfpower[ch]['calo'])
    nocalo_vals = np.array(lfpower[ch]['nocalo'])
    all_vals = np.concatenate([calo_vals, nocalo_vals])
    all_vals = all_vals[all_vals > 0]
    bins = np.logspace(np.log10(np.percentile(all_vals, 0.5)), np.log10(np.percentile(all_vals, 99.5)), 60)
    ax.hist(nocalo_vals, bins=bins, color='steelblue', alpha=0.6, label=f'calo assente (n={len(nocalo_vals)})')
    ax.hist(calo_vals, bins=bins, color='crimson', alpha=0.6, label=f'calo presente (n={len(calo_vals)})')
    ax.set_xscale('log')
    ax.set_xlabel(f'Potenza FFT integrata {LF_BAND_MHZ[0]:.0f}-{LF_BAND_MHZ[1]:.0f} MHz')
    ax.set_ylabel('eventi')
    ax.set_title(f'ch {lbl}')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
fig.suptitle(f'Run {RUN} — Potenza FFT bassa frequenza: calo presente vs assente', fontsize=12)
plt.tight_layout()
plt.show()

print()
for ch, lbl in [(CH_A, 'A'), (CH_B, 'B')]:
    calo_vals = np.array(lfpower[ch]['calo'])
    nocalo_vals = np.array(lfpower[ch]['nocalo'])

    # AUC via Mann-Whitney U: U / (n1*n2), 0.5=nessuna separazione, 1.0=perfetta
    u_stat, _ = mannwhitneyu(calo_vals, nocalo_vals, alternative='two-sided')
    auc = u_stat / (len(calo_vals) * len(nocalo_vals))

    # miglior taglio singolo (accuratezza bilanciata: media di sensibilita' e specificita')
    thr_candidates = np.linspace(min(calo_vals.min(), nocalo_vals.min()),
                                  max(calo_vals.max(), nocalo_vals.max()), 500)
    best_bal_acc, best_thr = 0.0, None
    for thr in thr_candidates:
        tpr = np.mean(calo_vals > thr)          # sensibilita': calo correttamente sopra soglia
        tnr = np.mean(nocalo_vals <= thr)        # specificita': no-calo correttamente sotto soglia
        bal_acc = 0.5 * (tpr + tnr)
        if bal_acc > best_bal_acc:
            best_bal_acc, best_thr = bal_acc, thr

    print(f"ch {lbl}: AUC = {auc:.3f}  |  miglior taglio singolo: soglia={best_thr:.1f}, "
          f"accuratezza bilanciata = {best_bal_acc*100:.1f}%")

## Spettro FFT su eventi che superano una soglia in ampiezza (100-200 ns)

Pre-selezione aggiuntiva rispetto allo studio FFT di base: un evento "sopravvive" solo se
il segnale A **oppure** B (PADME-corretto, baseline-sottratta) scende sotto `AMP_THR_MV`
nella finestra `AMP_WIN_NS` (100-200 ns). Per i soli eventi sopravvissuti si rifà lo stesso
spettro medio, separato per calo presente/assente — sulla stessa finestra temporale completa
usata per la FFT finora, non solo sulla finestra di selezione.

Utile per vedere: tra gli eventi che mostrano comunque un'escursione di ampiezza (con o senza
calo), lo spettro tra i due gruppi è ancora diverso, o l'aver già filtrato su un'escursione
reale rende i due gruppi più simili tra loro (es. se molti "calo assente sopravvissuti" sono
in realta' rumore che casualmente supera la soglia in ampiezza)?

In [ ]:
AMP_THR_MV = -1.0
AMP_THR_ADC = AMP_THR_MV / ADC_TO_MV
AMP_WIN_NS = (100.0, 200.0)
N_FFT2_EVENTS = 2716   # eventi da scansionare (tutti quelli disponibili nel run)

mfile_fft2 = wc.open_mid(run=RUN, path=path, cloud=CLOUD, tag=TAG, verbose=VERBOSE)

sum_mag2 = {ch: {w: {'calo': None, 'nocalo': None} for w in WINDOWS} for ch in (CH_A, CH_B)}
n_calo2, n_nocalo2, n_survived = 0, 0, 0
freqs_mhz2 = None
_hann_cache2 = {}

n_ev = 0
for event in mfile_fft2:
    if n_ev >= N_FFT2_EVENTS:
        break
    for bank_name, bank in event.banks.items():
        if 'DGH0' not in bank_name:
            continue
        header = wc.daq_dgz_full2header(bank, verbose=False)
        waveform = wc.daq_dgz_full2array(event.banks['DIG0'], header, verbose=False)  # PADME-corretto
        n_ch = int(header.nchannels[0]); n_sub = int(header.ntriggers[0])
        for isub in range(n_sub):
            if n_ev >= N_FFT2_EVENTS:
                break
            n_ev += 1
            spc = len(waveform[isub*n_ch]) - N_DROP_LAST_FFT
            t_ns = np.arange(spc) * dt_ns

            if spc not in _hann_cache2:
                _hann_cache2[spc] = np.hanning(spc)
            hann_win = _hann_cache2[spc]

            amp_mask = (t_ns >= AMP_WIN_NS[0]) & (t_ns <= AMP_WIN_NS[1])

            wfA = np.asarray(waveform[isub*n_ch+CH_A][:spc], dtype=float)
            blA = _interp_baseline(wfA, t_ns, PMT_BL_LEFT_NS, PMT_BL_RIGHT_NS, dt_ns)
            sigA = wfA - blA
            wfB = np.asarray(waveform[isub*n_ch+CH_B][:spc], dtype=float)
            blB = _interp_baseline(wfB, t_ns, PMT_BL_LEFT_NS, PMT_BL_RIGHT_NS, dt_ns)
            sigB = wfB - blB

            survives = (sigA[amp_mask].min() <= AMP_THR_ADC) or (sigB[amp_mask].min() <= AMP_THR_ADC)
            if not survives:
                continue
            n_survived += 1

            # --- classificazione calo (stesso metodo di WC_analisys_tool) ---
            wf_calo = np.asarray(waveform[isub*n_ch+CH_CALO][:spc], dtype=float)
            bl_calo = _interp_baseline(wf_calo, t_ns, CALO_BL_LEFT_NS, CALO_BL_RIGHT_NS, dt_ns)
            is_calo_valid = np.min(wf_calo - bl_calo) <= CALO_THR
            grp = 'calo' if is_calo_valid else 'nocalo'
            if is_calo_valid: n_calo2 += 1
            else: n_nocalo2 += 1

            for ch, sig in [(CH_A, sigA), (CH_B, sigB)]:
                mag_rect = np.abs(np.fft.rfft(sig))
                mag_hann = np.abs(np.fft.rfft(sig * hann_win))
                if freqs_mhz2 is None:
                    freqs_mhz2 = np.fft.rfftfreq(spc, d=dt_ns) * 1000.0
                for wname, mag in (('rect', mag_rect), ('hann', mag_hann)):
                    if sum_mag2[ch][wname][grp] is None:
                        sum_mag2[ch][wname][grp] = np.zeros_like(mag)
                    sum_mag2[ch][wname][grp] += mag
        break

print(f"Eventi analizzati: {n_ev}  |  sopravvissuti alla soglia {AMP_THR_MV} mV in {AMP_WIN_NS} ns: {n_survived}")
print(f"  di cui calo presente: {n_calo2}, calo assente: {n_nocalo2}")

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
for row, wname, wlabel in [(0, 'rect', 'nessuna finestra (rettangolare)'), (1, 'hann', 'finestra di Hann')]:
    for col, ch, lbl in [(0, CH_A, 'A'), (1, CH_B, 'B')]:
        ax = axes[row][col]
        mean_calo   = (sum_mag2[ch][wname]['calo']   / n_calo2)   if (sum_mag2[ch][wname]['calo']   is not None and n_calo2   > 0) else None
        mean_nocalo = (sum_mag2[ch][wname]['nocalo'] / n_nocalo2) if (sum_mag2[ch][wname]['nocalo'] is not None and n_nocalo2 > 0) else None
        if mean_calo is not None:
            ax.plot(freqs_mhz2, mean_calo, color='crimson', lw=1.2, label=f'calo presente (n={n_calo2})')
        if mean_nocalo is not None:
            ax.plot(freqs_mhz2, mean_nocalo, color='steelblue', lw=1.2, label=f'calo assente (n={n_nocalo2})')
        ax.set_yscale('log')
        ax.set_title(f'ch {lbl} — {wlabel}', fontsize=10)
        if row == 1:
            ax.set_xlabel('Frequenza (MHz)')
        if col == 0:
            ax.set_ylabel('|FFT| medio (ADC counts)')
        ax.grid(True, which='both', alpha=0.3); ax.legend(fontsize=8)
fig.suptitle(f'Run {RUN} — Spettro FFT (PADME-corretto), eventi con |ampiezza| > {abs(AMP_THR_MV):.0f} mV in {AMP_WIN_NS} ns',
             fontsize=12)
plt.tight_layout()
plt.show()

### Stesso studio, ma come accumulo di tutte le FFT individuali (non solo la media)

La media nasconde la variabilita' evento-per-evento: un singolo evento anomalo si perde
nella media di centinaia. Qui si accumula un istogramma 2D (frequenza vs ampiezza log)
di **ogni** FFT individuale, cosi' si vede la densita' reale della popolazione (dove si
concentrano gli eventi, e dove invece ci sono outlier isolati) invece di una sola curva
riassuntiva. La media (stessa di prima) e' sovrapposta in bianco per riferimento.

In [ ]:
mfile_fft2b = wc.open_mid(run=RUN, path=path, cloud=CLOUD, tag=TAG, verbose=VERBOSE)

all_spectra = {ch: {w: {'calo': [], 'nocalo': []} for w in WINDOWS} for ch in (CH_A, CH_B)}
n_calo2b, n_nocalo2b = 0, 0
freqs_mhz2b = None
_hann_cache2b = {}

n_ev = 0
for event in mfile_fft2b:
    if n_ev >= N_FFT2_EVENTS:
        break
    for bank_name, bank in event.banks.items():
        if 'DGH0' not in bank_name:
            continue
        header = wc.daq_dgz_full2header(bank, verbose=False)
        waveform = wc.daq_dgz_full2array(event.banks['DIG0'], header, verbose=False)  # PADME-corretto
        n_ch = int(header.nchannels[0]); n_sub = int(header.ntriggers[0])
        for isub in range(n_sub):
            if n_ev >= N_FFT2_EVENTS:
                break
            n_ev += 1
            spc = len(waveform[isub*n_ch]) - N_DROP_LAST_FFT
            t_ns = np.arange(spc) * dt_ns

            if spc not in _hann_cache2b:
                _hann_cache2b[spc] = np.hanning(spc)
            hann_win = _hann_cache2b[spc]

            amp_mask = (t_ns >= AMP_WIN_NS[0]) & (t_ns <= AMP_WIN_NS[1])

            wfA = np.asarray(waveform[isub*n_ch+CH_A][:spc], dtype=float)
            blA = _interp_baseline(wfA, t_ns, PMT_BL_LEFT_NS, PMT_BL_RIGHT_NS, dt_ns)
            sigA = wfA - blA
            wfB = np.asarray(waveform[isub*n_ch+CH_B][:spc], dtype=float)
            blB = _interp_baseline(wfB, t_ns, PMT_BL_LEFT_NS, PMT_BL_RIGHT_NS, dt_ns)
            sigB = wfB - blB

            survives = (sigA[amp_mask].min() <= AMP_THR_ADC) or (sigB[amp_mask].min() <= AMP_THR_ADC)
            if not survives:
                continue

            wf_calo = np.asarray(waveform[isub*n_ch+CH_CALO][:spc], dtype=float)
            bl_calo = _interp_baseline(wf_calo, t_ns, CALO_BL_LEFT_NS, CALO_BL_RIGHT_NS, dt_ns)
            is_calo_valid = np.min(wf_calo - bl_calo) <= CALO_THR
            grp = 'calo' if is_calo_valid else 'nocalo'
            if is_calo_valid: n_calo2b += 1
            else: n_nocalo2b += 1

            for ch, sig in [(CH_A, sigA), (CH_B, sigB)]:
                mag_rect = np.abs(np.fft.rfft(sig))
                mag_hann = np.abs(np.fft.rfft(sig * hann_win))
                if freqs_mhz2b is None:
                    freqs_mhz2b = np.fft.rfftfreq(spc, d=dt_ns) * 1000.0
                all_spectra[ch]['rect'][grp].append(mag_rect)
                all_spectra[ch]['hann'][grp].append(mag_hann)
        break

print(f"Eventi accumulati: calo presente={n_calo2b}, calo assente={n_nocalo2b}")

for grp, grp_label in [('calo', 'calo presente'), ('nocalo', 'calo assente')]:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
    for row, wname, wlabel in [(0, 'rect', 'nessuna finestra (rettangolare)'), (1, 'hann', 'finestra di Hann')]:
        for col, ch, lbl in [(0, CH_A, 'A'), (1, CH_B, 'B')]:
            ax = axes[row][col]
            arr = np.array(all_spectra[ch][wname][grp])   # shape (n_eventi, n_freq_bins)
            if arr.size == 0:
                continue
            n_events_g = arr.shape[0]
            freq_grid = np.tile(freqs_mhz2b, n_events_g)
            amp_grid = np.clip(arr.flatten(), 1.0, None)
            h = ax.hist2d(freq_grid, np.log10(amp_grid), bins=[120, 80], cmap='viridis', cmin=1)
            fig.colorbar(h[3], ax=ax, label='n. eventi')
            mean_curve = arr.mean(axis=0)
            ax.plot(freqs_mhz2b, np.log10(np.clip(mean_curve, 1.0, None)), color='white', lw=1.3, label='media')
            ax.set_title(f'ch {lbl} — {wlabel} (n={n_events_g})', fontsize=10)
            if row == 1:
                ax.set_xlabel('Frequenza (MHz)')
            if col == 0:
                ax.set_ylabel('log10(|FFT|)')
            ax.legend(fontsize=8, loc='upper right')
    fig.suptitle(f'Run {RUN} — Accumulo FFT individuali ({grp_label}), soglia {AMP_THR_MV} mV in {AMP_WIN_NS} ns',
                 fontsize=12)
    plt.tight_layout()
    plt.show()

## Studio MC: FFT gamma (Compton) vs mu, nessuna soglia

Confronto sui segnali digitalizzati MC (`Analysis_script`/`Signal_maker.ipynb`, file `.npz` in
`MC_DIR`, non i `.mid.gz` reali usati finora — qui non c'e' correzione PADME da applicare,
ne' un canale calo: la sorgente e' nota per file, non va classificata evento per evento).

File usati (stessa geometria `single`, stessa posizione X0/Y0/Th0):
- `sim_single_gamma_X0mm_Y0mm_Th0deg.npz` — segnale da elettroni Compton (sorgente gamma)
- `sim_single_mu_tb_X0mm_Y0mm_Th0deg.npz` — segnale da mu diretti (sorgente mu testbeam;
  `mu_lngs` locale ha solo 5 eventi, non abbastanza per una media)

Nessuna soglia sui segnali (come richiesto): si accumula la FFT di **tutti** gli eventi di
ciascun file, media per canale, e si confrontano le due sorgenti direttamente.

In [ ]:
MC_DIR = '/Users/benussi/Testbeam2026_WC_unified/data'
MC_FILE_GAMMA = os.path.join(MC_DIR, 'sim_single_gamma_X0mm_Y0mm_Th0deg.npz')
MC_FILE_MU    = os.path.join(MC_DIR, 'sim_single_mu_tb_X0mm_Y0mm_Th0deg.npz')

MC_CH_A, MC_CH_B = 0, 1   # ch_0, ch_1 nei file MC (PMT sx/dx)
dt_ns_mc = 0.4
spc_mc = 1024
t_ns_mc = np.arange(spc_mc) * dt_ns_mc

def load_mc_events(path):
    d = np.load(path, allow_pickle=True)
    return [d[k].item() for k in d.keys()]

gamma_events = load_mc_events(MC_FILE_GAMMA)
mu_events    = load_mc_events(MC_FILE_MU)
n_gamma, n_mu = len(gamma_events), len(mu_events)
print(f"gamma: {n_gamma} eventi  |  mu_tb: {n_mu} eventi")

sum_mag_mc = {ch: {'gamma': np.zeros(spc_mc//2 + 1), 'mu': np.zeros(spc_mc//2 + 1)} for ch in (MC_CH_A, MC_CH_B)}
freqs_mhz_mc = np.fft.rfftfreq(spc_mc, d=dt_ns_mc) * 1000.0

for src_key, events in [('gamma', gamma_events), ('mu', mu_events)]:
    for ev in events:
        for ch in (MC_CH_A, MC_CH_B):
            wf = ev[f'ch_{ch}'].astype(float)
            bl = _interp_baseline(wf, t_ns_mc, PMT_BL_LEFT_NS, PMT_BL_RIGHT_NS, dt_ns_mc)
            sig = wf - bl
            mag = np.abs(np.fft.rfft(sig))
            sum_mag_mc[ch][src_key] += mag

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, ch, lbl in [(axes[0], MC_CH_A, 'A (ch_0)'), (axes[1], MC_CH_B, 'B (ch_1)')]:
    mean_gamma = sum_mag_mc[ch]['gamma'] / n_gamma
    mean_mu    = sum_mag_mc[ch]['mu'] / n_mu
    ax.plot(freqs_mhz_mc, mean_gamma, color='purple',    lw=1.2, label=f'gamma / Compton (n={n_gamma})')
    ax.plot(freqs_mhz_mc, mean_mu,    color='darkorange', lw=1.2, label=f'mu_tb (n={n_mu})')
    ax.set_yscale('log')
    ax.set_xlabel('Frequenza (MHz)'); ax.set_ylabel('|FFT| medio (ADC counts)')
    ax.set_title(f'ch {lbl}')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
fig.suptitle('MC (single, X0 Y0 Th0) — Spettro FFT medio: gamma (Compton) vs mu_tb, nessuna soglia',
             fontsize=12)
plt.tight_layout()
plt.show()

### Stesso confronto, ma con segnale NORMALIZZATO al proprio picco (forma, non ampiezza)

Ogni evento viene diviso per il proprio picco (`|min|` del segnale baseline-sottratto) prima
della FFT, cosi' ogni impulso ha ampiezza 1 indipendentemente da quanto era grande in origine.
Se le curve gamma/mu qui si sovrappongono (a differenza del plot precedente), la differenza
vista prima era solo di ampiezza; se restano diverse, e' una vera differenza di forma/frequenza.

Eventi con picco troppo vicino a zero (`< MIN_AMP_FOR_NORM`) sono esclusi per evitare di
normalizzare rumore puro (instabile: dividere per un numero vicino a zero amplifica tutto).

In [ ]:
MIN_AMP_FOR_NORM = 5.0   # ADC counts, evita normalizzazioni instabili su eventi quasi senza picco

sum_mag_mc_norm = {ch: {'gamma': np.zeros(spc_mc//2 + 1), 'mu': np.zeros(spc_mc//2 + 1)} for ch in (MC_CH_A, MC_CH_B)}
n_used = {'gamma': 0, 'mu': 0}

for src_key, events in [('gamma', gamma_events), ('mu', mu_events)]:
    for ev in events:
        per_ch_sig = {}
        ok_event = True
        for ch in (MC_CH_A, MC_CH_B):
            wf = ev[f'ch_{ch}'].astype(float)
            bl = _interp_baseline(wf, t_ns_mc, PMT_BL_LEFT_NS, PMT_BL_RIGHT_NS, dt_ns_mc)
            sig = wf - bl
            peak = -sig.min()   # positivo, il segnale scende sotto la baseline
            if peak < MIN_AMP_FOR_NORM:
                ok_event = False
                break
            per_ch_sig[ch] = sig / peak   # normalizzato: picco = 1 per QUESTO canale
        if not ok_event:
            continue
        n_used[src_key] += 1
        for ch in (MC_CH_A, MC_CH_B):
            mag = np.abs(np.fft.rfft(per_ch_sig[ch]))
            sum_mag_mc_norm[ch][src_key] += mag

print(f"Eventi usati dopo normalizzazione (picco >= {MIN_AMP_FOR_NORM:.0f} ADC su entrambi i canali): "
      f"gamma={n_used['gamma']}/{n_gamma}, mu_tb={n_used['mu']}/{n_mu}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, ch, lbl in [(axes[0], MC_CH_A, 'A (ch_0)'), (axes[1], MC_CH_B, 'B (ch_1)')]:
    mean_gamma = sum_mag_mc_norm[ch]['gamma'] / n_used['gamma']
    mean_mu    = sum_mag_mc_norm[ch]['mu'] / n_used['mu']
    ax.plot(freqs_mhz_mc, mean_gamma, color='purple',    lw=1.2, label=f"gamma (n={n_used['gamma']})")
    ax.plot(freqs_mhz_mc, mean_mu,    color='darkorange', lw=1.2, label=f"mu_tb (n={n_used['mu']})")
    ax.set_yscale('log')
    ax.set_xlabel('Frequenza (MHz)'); ax.set_ylabel('|FFT| medio (segnale normalizzato al picco)')
    ax.set_title(f'ch {lbl}')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
fig.suptitle('MC (single, X0 Y0 Th0) — Spettro FFT medio NORMALIZZATO al picco: gamma vs mu_tb',
             fontsize=12)
plt.tight_layout()
plt.show()

### Confronto gamma vs mu con soglia di ampiezza sui segnali MC (segnale grezzo, non normalizzato)

Invece di normalizzare al picco (che confonde forma e rapporto segnale/rumore, vedi sopra),
si applica una **soglia di ampiezza** `MC_AMP_THR_MV` (mV, convertita in ADC con `ADC_TO_MV`
gia' definito nella cella di CONFIG) nella finestra segnale `MC_AMP_WIN_NS` = (100, 300) ns
(la regione tra le due finestre di baseline). Un evento sopravvive se almeno uno dei due canali
(A o B) supera la soglia in quella finestra. Si confronta poi la FFT media (grezza, in ADC) dei
soli eventi sopravvissuti tra gamma e mu_tb -- stesso approccio gia' usato sui DATI reali piu'
sopra in questo notebook. `MC_AMP_THR_MV` e' facilmente modificabile per esplorare altre soglie.


In [ ]:
MC_AMP_THR_MV = -3.0   # soglia sui segnali MC (mV) -- cambiare qui e ri-eseguire per provare altre soglie
MC_AMP_THR_ADC = MC_AMP_THR_MV / ADC_TO_MV
MC_AMP_WIN_NS = (100.0, 200.0)   # finestra segnale (dove sono i picchi reali)

_mask_win_mc = (t_ns_mc >= MC_AMP_WIN_NS[0]) & (t_ns_mc <= MC_AMP_WIN_NS[1])

def mc_event_peaks(ev):
    peaks = []
    for ch in (MC_CH_A, MC_CH_B):
        wf = ev[f'ch_{ch}'].astype(float)
        bl = _interp_baseline(wf, t_ns_mc, PMT_BL_LEFT_NS, PMT_BL_RIGHT_NS, dt_ns_mc)
        sig = wf - bl
        peaks.append(sig[_mask_win_mc].min())
    return peaks   # negativi (dip sotto baseline)

sum_mag_mc_thr = {ch: {'gamma': np.zeros(spc_mc//2 + 1), 'mu': np.zeros(spc_mc//2 + 1)} for ch in (MC_CH_A, MC_CH_B)}
n_surv = {'gamma': 0, 'mu': 0}

for src_key, events in [('gamma', gamma_events), ('mu', mu_events)]:
    for ev in events:
        pA, pB = mc_event_peaks(ev)
        if min(pA, pB) > MC_AMP_THR_ADC:   # nessuno dei due canali supera la soglia
            continue
        n_surv[src_key] += 1
        for ch in (MC_CH_A, MC_CH_B):
            wf = ev[f'ch_{ch}'].astype(float)
            bl = _interp_baseline(wf, t_ns_mc, PMT_BL_LEFT_NS, PMT_BL_RIGHT_NS, dt_ns_mc)
            sig = wf - bl
            mag = np.abs(np.fft.rfft(sig))
            sum_mag_mc_thr[ch][src_key] += mag

print(f"Soglia: {MC_AMP_THR_MV:+.1f} mV ({MC_AMP_THR_ADC:+.1f} ADC) in {MC_AMP_WIN_NS} ns  ->  "
      f"gamma superstiti: {n_surv['gamma']}/{n_gamma} ({100*n_surv['gamma']/n_gamma:.1f}%)   "
      f"mu_tb superstiti: {n_surv['mu']}/{n_mu} ({100*n_surv['mu']/n_mu:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, ch, lbl in [(axes[0], MC_CH_A, 'A (ch_0)'), (axes[1], MC_CH_B, 'B (ch_1)')]:
    mean_gamma = sum_mag_mc_thr[ch]['gamma'] / max(n_surv['gamma'], 1)
    mean_mu    = sum_mag_mc_thr[ch]['mu'] / max(n_surv['mu'], 1)
    ax.plot(freqs_mhz_mc, mean_gamma, color='purple',    lw=1.2, label=f"gamma (n={n_surv['gamma']})")
    ax.plot(freqs_mhz_mc, mean_mu,    color='darkorange', lw=1.2, label=f"mu_tb (n={n_surv['mu']})")
    ax.set_yscale('log')
    ax.set_xlabel('Frequenza (MHz)'); ax.set_ylabel('|FFT| medio (ADC counts)')
    ax.set_title(f'ch {lbl}')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
fig.suptitle(f"MC -- Spettro FFT medio (segnale grezzo): gamma vs mu_tb, soglia {MC_AMP_THR_MV:+.1f} mV",
             fontsize=12)
plt.tight_layout()
plt.show()


### Visualizzazione dei singoli eventi gamma-Compton superstiti alla soglia

Mostra le forme d'onda individuali (non mediate) dei primi `N_SHOW_GAMMA_WF` eventi gamma che
sopravvivono alla soglia `MC_AMP_THR_MV` nella finestra `MC_AMP_WIN_NS` (stessi parametri della
cella precedente) -- per vedere direttamente se sono impulsi reali o fluttuazioni di rumore.
Ogni pannello mostra i due canali A (ch_0, rosso) e B (ch_1, nero), con la soglia e la finestra
di selezione evidenziate.


In [ ]:
N_SHOW_GAMMA_WF = 10000   # numero di eventi gamma superstiti da mostrare (soft limit, vedi _MAX_PANELS)
_MAX_PANELS = 30         # tetto di sicurezza sul numero di pannelli disegnati

_n_survivors_gamma = sum(1 for ev in gamma_events if min(mc_event_peaks(ev)) <= MC_AMP_THR_ADC)
_n_panels_gamma = min(N_SHOW_GAMMA_WF, _n_survivors_gamma, _MAX_PANELS)
_ncols_g = 3
_nrows_g = max(1, int(np.ceil(_n_panels_gamma / _ncols_g)))

_shown = 0
_fig, _axes = plt.subplots(_nrows_g, _ncols_g, figsize=(15, 4*_nrows_g), sharex=True)
_axes = np.atleast_1d(_axes).ravel()

for ev in gamma_events:
    if _shown >= _n_panels_gamma:
        break
    pA, pB = mc_event_peaks(ev)
    if min(pA, pB) > MC_AMP_THR_ADC:
        continue

    wfA = ev[f'ch_{MC_CH_A}'].astype(float)
    blA = _interp_baseline(wfA, t_ns_mc, PMT_BL_LEFT_NS, PMT_BL_RIGHT_NS, dt_ns_mc)
    sigA_mv = (wfA - blA) * ADC_TO_MV

    wfB = ev[f'ch_{MC_CH_B}'].astype(float)
    blB = _interp_baseline(wfB, t_ns_mc, PMT_BL_LEFT_NS, PMT_BL_RIGHT_NS, dt_ns_mc)
    sigB_mv = (wfB - blB) * ADC_TO_MV

    ax = _axes[_shown]
    ax.plot(t_ns_mc, sigA_mv, color='red', lw=1.0, label='A (ch_0)')
    ax.plot(t_ns_mc, sigB_mv, color='black', lw=1.0, label='B (ch_1)')
    ax.axhline(MC_AMP_THR_MV, color='gray', ls='--', lw=0.8)
    ax.axvspan(MC_AMP_WIN_NS[0], MC_AMP_WIN_NS[1], color='yellow', alpha=0.15)
    ax.set_title(f"peakA={pA*ADC_TO_MV:.1f}mV peakB={pB*ADC_TO_MV:.1f}mV", fontsize=9)
    ax.set_xlim(100, 300)
    ax.grid(alpha=0.3)
    if _shown == 0:
        ax.legend(fontsize=8)
    _shown += 1

for _i in range(_shown, len(_axes)):
    _axes[_i].axis('off')

_fig.suptitle(f"Eventi gamma-Compton superstiti (soglia {MC_AMP_THR_MV:+.1f} mV in {MC_AMP_WIN_NS} ns) -- mostrati {_shown}/{_n_survivors_gamma} superstiti",
              fontsize=12)
_fig.supxlabel('Tempo (ns)')
_fig.supylabel('Segnale (mV, baseline sottratta)')
plt.tight_layout()
plt.show()


### Confronto: stessi eventi ma per i mu_tb (test beam) -- ROI temporale completa

Stessa selezione (soglia `MC_AMP_THR_MV` in `MC_AMP_WIN_NS`) applicata alla popolazione mu_tb,
per confronto diretto con gli eventi gamma-Compton mostrati sopra. Qui l'asse temporale resta
sulla ROI completa (0-410 ns, tutta la traccia) e non zoomato, a differenza del plot gamma-Compton:
l'impulso mu_tb e' gia' ben visibile senza bisogno di zoom.


In [ ]:
N_SHOW_MU_WF = 12   # numero di eventi mu_tb superstiti da mostrare (soft limit, vedi _MAX_PANELS)
_MAX_PANELS_MU = 30   # tetto di sicurezza sul numero di pannelli disegnati

_n_survivors_mu = sum(1 for ev in mu_events if min(mc_event_peaks(ev)) <= MC_AMP_THR_ADC)
_n_panels_mu = min(N_SHOW_MU_WF, _n_survivors_mu, _MAX_PANELS_MU)
_ncols_m = 3
_nrows_m = max(1, int(np.ceil(_n_panels_mu / _ncols_m)))

_shown_mu = 0
_fig, _axes = plt.subplots(_nrows_m, _ncols_m, figsize=(15, 4*_nrows_m), sharex=True)
_axes = np.atleast_1d(_axes).ravel()

for ev in mu_events:
    if _shown_mu >= _n_panels_mu:
        break
    pA, pB = mc_event_peaks(ev)
    if min(pA, pB) > MC_AMP_THR_ADC:
        continue

    wfA = ev[f'ch_{MC_CH_A}'].astype(float)
    blA = _interp_baseline(wfA, t_ns_mc, PMT_BL_LEFT_NS, PMT_BL_RIGHT_NS, dt_ns_mc)
    sigA_mv = (wfA - blA) * ADC_TO_MV

    wfB = ev[f'ch_{MC_CH_B}'].astype(float)
    blB = _interp_baseline(wfB, t_ns_mc, PMT_BL_LEFT_NS, PMT_BL_RIGHT_NS, dt_ns_mc)
    sigB_mv = (wfB - blB) * ADC_TO_MV

    ax = _axes[_shown_mu]
    ax.plot(t_ns_mc, sigA_mv, color='red', lw=1.0, label='A (ch_0)')
    ax.plot(t_ns_mc, sigB_mv, color='black', lw=1.0, label='B (ch_1)')
    ax.axhline(MC_AMP_THR_MV, color='gray', ls='--', lw=0.8)
    ax.axvspan(MC_AMP_WIN_NS[0], MC_AMP_WIN_NS[1], color='yellow', alpha=0.15)
    ax.set_title(f"peakA={pA*ADC_TO_MV:.1f}mV peakB={pB*ADC_TO_MV:.1f}mV", fontsize=9)
    ax.set_xlim(0, 410)   # ROI temporale completa (a differenza del plot gamma-Compton, qui non zoomato)
    ax.grid(alpha=0.3)
    if _shown_mu == 0:
        ax.legend(fontsize=8)
    _shown_mu += 1

for _i in range(_shown_mu, len(_axes)):
    _axes[_i].axis('off')

_fig.suptitle(f"Eventi mu_tb superstiti (soglia {MC_AMP_THR_MV:+.1f} mV in {MC_AMP_WIN_NS} ns) -- mostrati {_shown_mu}/{_n_survivors_mu} superstiti")
_fig.supxlabel('Tempo (ns)')
_fig.supylabel('Segnale (mV, baseline sottratta)')
plt.tight_layout()
plt.show()


## Ispezione visiva dei 29 eventi "calo assente" che superano la soglia in ampiezza

Mostra ogni evento (PMT A/B, Trigger, e stavolta il vero Calorimetro con `CALO_THR` marcato,
non il placeholder) per verificare a occhio se la forma d'onda assomiglia a un impulso fisico
reale e se il calorimetro era "vicino" a validare l'evento o del tutto piatto.

In [ ]:
mfile_show29 = wc.open_mid(run=RUN, path=path, cloud=CLOUD, tag=TAG, verbose=VERBOSE)

MAX_SHOW29 = 50   # sicurezza, sappiamo che sono ~29 nel run
n_ev = 0
n_shown29 = 0
n_abs29 = 0

for event in mfile_show29:
    if n_shown29 >= MAX_SHOW29 or n_ev >= N_FFT2_EVENTS:
        break
    for bank_name, bank in event.banks.items():
        if 'DGH0' not in bank_name:
            continue
        header = wc.daq_dgz_full2header(bank, verbose=False)
        waveform = wc.daq_dgz_full2array(event.banks['DIG0'], header, verbose=False)  # PADME-corretto
        n_ch = int(header.nchannels[0]); n_sub = int(header.ntriggers[0])
        spc_full = len(waveform[0])
        spc = spc_full - N_DROP_LAST_FFT
        t_ns = np.arange(spc) * dt_ns
        mask_win = (t_ns >= WINDOW_NS[0]) & (t_ns <= WINDOW_NS[1])
        amp_mask = (t_ns >= AMP_WIN_NS[0]) & (t_ns <= AMP_WIN_NS[1])

        for isub in range(n_sub):
            if n_shown29 >= MAX_SHOW29 or n_ev >= N_FFT2_EVENTS:
                break
            n_ev += 1
            n_abs29 += 1

            wfA = np.asarray(waveform[isub*n_ch+CH_A][:spc], dtype=float)
            blA = _interp_baseline(wfA, t_ns, PMT_BL_LEFT_NS, PMT_BL_RIGHT_NS, dt_ns)
            sigA = wfA - blA
            wfB = np.asarray(waveform[isub*n_ch+CH_B][:spc], dtype=float)
            blB = _interp_baseline(wfB, t_ns, PMT_BL_LEFT_NS, PMT_BL_RIGHT_NS, dt_ns)
            sigB = wfB - blB

            survives = (sigA[amp_mask].min() <= AMP_THR_ADC) or (sigB[amp_mask].min() <= AMP_THR_ADC)
            if not survives:
                continue

            wf_calo = np.asarray(waveform[isub*n_ch+CH_CALO][:spc], dtype=float)
            bl_calo = _interp_baseline(wf_calo, t_ns, CALO_BL_LEFT_NS, CALO_BL_RIGHT_NS, dt_ns)
            sig_calo = wf_calo - bl_calo
            is_calo_valid = np.min(sig_calo) <= CALO_THR
            if is_calo_valid:
                continue   # ci interessano solo i "calo assente"

            sic = int(header.SIC[0][isub]) if len(header.SIC) > 0 else None
            sic_t_ns = sic * dt_ns if sic is not None else None

            fig, axes = plt.subplots(1, 3, figsize=(21, 5))

            ax0 = axes[0]
            ax0.plot(t_ns[mask_win], sigA[mask_win] * ADC_TO_MV, color='red',   lw=1.5, label=f'PMT A (ch {CH_A})')
            ax0.plot(t_ns[mask_win], sigB[mask_win] * ADC_TO_MV, color='black', lw=1.5, label=f'PMT B (ch {CH_B})')
            ax0.axhline(AMP_THR_MV, color='gray', ls=':', alpha=0.6, label=f'soglia {AMP_THR_MV} mV')
            ax0.axhline(0, color='gray', lw=0.8, ls='--')
            ax0.axvspan(AMP_WIN_NS[0], AMP_WIN_NS[1], color='gold', alpha=0.15, label='finestra selezione')
            if sic_t_ns is not None and WINDOW_NS[0] <= sic_t_ns <= WINDOW_NS[1]:
                ax0.axvline(sic_t_ns, color='navy', ls='--', lw=1.2, alpha=0.8, label=f'SIC = {sic_t_ns:.1f} ns')
            ax0.set_title("PMT Signals (correzione PADME)")
            ax0.set_xlabel("Time (ns)"); ax0.set_ylabel("Amplitude (mV)")
            ax0.set_xlim(*WINDOW_NS)
            ax0.grid(True, ls='--', alpha=0.5); ax0.legend(loc='best', fontsize=8)

            ax1 = axes[1]
            wfT = np.asarray(waveform[isub*n_ch+CH_TRIG][:spc], dtype=float)
            blT = _interp_baseline(wfT, t_ns, PMT_BL_LEFT_NS, PMT_BL_RIGHT_NS, dt_ns)
            sigT = wfT - blT
            ax1.plot(t_ns[mask_win], sigT[mask_win] * ADC_TO_MV, color='darkorange', lw=1.5,
                     label=f'Trigger (ch {CH_TRIG})')
            ax1.axhline(0, color='gray', lw=0.8, ls='--')
            ax1.set_title("Trigger (correzione PADME)")
            ax1.set_xlabel("Time (ns)"); ax1.set_ylabel("Amplitude (mV)")
            ax1.set_xlim(*WINDOW_NS)
            ax1.grid(True, ls='--', alpha=0.5); ax1.legend(loc='best', fontsize=8)

            ax2 = axes[2]
            ax2.plot(t_ns, sig_calo * ADC_TO_MV, color='green', lw=1.5, label=f'Calorimeter (ch {CH_CALO})')
            ax2.axhline(CALO_THR * ADC_TO_MV, color='green', ls='--', alpha=0.7,
                        label=f'CALO_THR={CALO_THR*ADC_TO_MV:.1f} mV')
            ax2.axvspan(CALO_BL_LEFT_NS[0], CALO_BL_LEFT_NS[1], color='yellow', alpha=0.2)
            ax2.axvspan(CALO_BL_RIGHT_NS[0], CALO_BL_RIGHT_NS[1], color='yellow', alpha=0.2)
            ax2.set_title(f"Calorimeter (min={sig_calo.min()*ADC_TO_MV:.1f} mV)")
            ax2.set_xlabel("Time (ns)"); ax2.set_ylabel("Amplitude (mV)")
            ax2.grid(True, ls='--', alpha=0.5); ax2.legend(loc='best', fontsize=8)

            fig.suptitle(f"Run {RUN} - AbsIdx: {n_abs29} | calo assente, sopravvive soglia ampiezza "
                         f"({AMP_THR_MV} mV in {AMP_WIN_NS} ns) — SIC = {sic_t_ns:.1f} ns", fontsize=12)
            plt.tight_layout()
            plt.show()
            plt.close(fig)

            n_shown29 += 1
        break   # una sola bank DGH0 per evento MIDAS

print(f"\nMostrati {n_shown29} eventi 'calo assente' che superano la soglia in ampiezza "
      f"(scansionati {n_ev} eventi totali).")